In [20]:
library(clusterProfiler)
suppressPackageStartupMessages(library(tableHTML))
library(ggplot2)
library(stringr)
library(org.Hs.eg.db)

In [2]:
td <- read.table("/tscc//projects/ps-epigen/users/rlan/Liver_FNIH/Scenic/Celltype_v2/Hepatocytes_out/scplus_pipeline_otsu/Snakemake/eRegulon_direct.tsv", header = TRUE)
#td <- read.table("/tscc//projects/ps-epigen/users/rlan/Liver_FNIH/Scenic/Celltype_v2/Hepatocytes_out/scplus_pipeline_yen/Snakemake/eRegulon_direct.tsv", header = TRUE)

td_sub <- td[,c("eRegulon_name", "Region")]
tdg <- td_sub$Region
head(td_sub)
length(unique(tdg))

,eRegulon_name,Region
,<chr>,<chr>
1,AR_direct_+/+,chr4:73965001-73965301
2,AR_direct_+/+,chr3:189383753-189384053
3,AR_direct_+/+,chr16:30844748-30845048
4,AR_direct_+/+,chr8:26693755-26694055
5,AR_direct_+/+,chr4:68963470-68963770
6,AR_direct_+/+,chrX:53086539-53086808


[1] 19388

In [3]:
tde <- td[td$TF == 'RELB',]
nrow(tde)

[1] 108

In [5]:
#tde

In [8]:
universe <- read.table("Hepatocytes_out/Seurat_data/Hepatocytes_region_names.tsv")
nrow(universe)
head(universe)

[1] 329796

,V1
,<chr>
1,chr1-16085-16372
2,chr1-17357-17657
3,chr1-29216-29497
4,chr1-102789-103089
5,chr1-136647-136947
6,chr1-138882-139182


In [9]:
phyper_grn <- function(grn_table, degs, universe, name){
    grns <- unique(grn_table$eRegulon_name)
    #print(length(grns))

    pvals <- c()
    for (g in grns){

        grn_gl <- grn_table[grn_table["eRegulon_name"] == g,]$Region
      #  grn_gl <- grn_table[grn_table["eRegulon_name"] == "KLF6_direct_+/-",]$Gene
        
        
        n_grn_gene <- length(grn_gl)
        n_intersect <- length(intersect(degs, grn_gl))
        n_canidates <- 329796
        n_deg <- length(degs)

    #    print(n_grn_gene)
    #    print(n_intersect)
    #    print(n_canidates)
    #    print(n_deg)
        
        #phyper(#n_over-laps, n_deg_size, n_all_canidate_genes, #n_genes_in_grn)
        hpval <- 1 - phyper(n_intersect, n_deg, n_canidates, n_grn_gene)
        pvals <- append(pvals, hpval)
    }
    qvals <- p.adjust(pvals, method="BH")

    cname <- str_remove(string = name, pattern = ".dds.res")
    cname_p <- paste0(cname, "_pval")
    cname_q <- paste0(cname, "_padj")
    
    
    pval_df <- DataFrame(row.names = grns)
    pval_df[cname_p] <- pvals
    pval_df[cname_q] <- qvals

    return(pval_df)
    

}

In [20]:
de_path <- "/tscc/lustre/ddn/scratch/welison/Liver_Share/result.depot/241213_WE_QTL_Feature_Beds/"
files <- list.files(de_path)
files <- grep("ATAC", files, value = TRUE)
files <- grep("fdr", files, value = TRUE)
files <- append(files, "Hepatocyte_ATAC_coloc_features.bed")
files

[1] "ATAC.B.feature.fdr.bed"             "ATAC.Bulk.feature.fdr.bed"         
 [3] "ATAC.Cholangiocyte.feature.fdr.bed" "ATAC.Endothelial.feature.fdr.bed"  
 [5] "ATAC.Hepatocytes.feature.fdr.bed"   "ATAC.HSC.feature.fdr.bed"          
 [7] "ATAC.Myeloid.feature.fdr.bed"       "ATAC.NK.feature.fdr.bed"           
 [9] "ATAC.T.feature.fdr.bed"             "Hepatocyte_ATAC_coloc_features.bed"

In [10]:
#t <- read.table(paste0(de_path, 'ATAC.Hepatocytes.feature.bed'), header = FALSE)
t <- read.table("/tscc/lustre/ddn/scratch/welison/Liver_Share/result.depot/241213_WE_QTL_Feature_Beds/ATAC.Hepatocytes.feature.fdr.bed", header = FALSE)
t <- na.omit(t)

In [11]:
#head(t, n  = 20)

In [12]:
regions <- paste0(t$V1, ":", t$V2, "-", t$V3)
head(regions)
length(regions)

[1] "chr1:1005257-1005557"     "chr1:100603502-100603766"
[3] "chr1:100623512-100623812" "chr1:100657913-100658126"
[5] "chr1:100658228-100658528" "chr1:101150736-101151036"

[1] 6106

In [17]:
length(intersect(regions, td_sub$Region))

[1] 490

In [13]:
res <- phyper_grn(grn_table = td_sub, degs = regions, universe = rownames(t), name = "ATAC.Hepatocytes")

[1] 1
[1] 16
[1] 2
[1] 4
[1] 1
[1] 35
[1] 22
[1] 1
[1] 1
[1] 2
[1] 5
[1] 11
[1] 2
[1] 6
[1] 3
[1] 1
[1] 5
[1] 1
[1] 2
[1] 1
[1] 15
[1] 1
[1] 2
[1] 12
[1] 3
[1] 7
[1] 9
[1] 1
[1] 1
[1] 3
[1] 7
[1] 5
[1] 2
[1] 0
[1] 0
[1] 2
[1] 2
[1] 2
[1] 17
[1] 2
[1] 20
[1] 12
[1] 0
[1] 4
[1] 10
[1] 0
[1] 1
[1] 1
[1] 13
[1] 18
[1] 3
[1] 0
[1] 18
[1] 1
[1] 3
[1] 11
[1] 8
[1] 0
[1] 2
[1] 1
[1] 8
[1] 0
[1] 1
[1] 7
[1] 19
[1] 11
[1] 5
[1] 1
[1] 19
[1] 7
[1] 2
[1] 10
[1] 2
[1] 2
[1] 0
[1] 2
[1] 0
[1] 13
[1] 6
[1] 5
[1] 0
[1] 12
[1] 0
[1] 5
[1] 9
[1] 0
[1] 3
[1] 32
[1] 34
[1] 7
[1] 0
[1] 2
[1] 3
[1] 0
[1] 8
[1] 2
[1] 30
[1] 15
[1] 12
[1] 5
[1] 12
[1] 1
[1] 0
[1] 2
[1] 0
[1] 3
[1] 3
[1] 0
[1] 17
[1] 1
[1] 0
[1] 14
[1] 1
[1] 0
[1] 0
[1] 1
[1] 2
[1] 1
[1] 1
[1] 0
[1] 0
[1] 3
[1] 0
[1] 0
[1] 0
[1] 2
[1] 1
[1] 0
[1] 0
[1] 0
[1] 0
[1] 2
[1] 0
[1] 0
[1] 0
[1] 1
[1] 0
[1] 0
[1] 1
[1] 1
[1] 1
[1] 0
[1] 1
[1] 0
[1] 2
[1] 0
[1] 1
[1] 0
[1] 0
[1] 1
[1] 4
[1] 0
[1] 1
[1] 0
[1] 1
[1] 1
[1] 0
[1] 0
[1] 1
[1] 1
[1] 0
[1] 1


In [14]:
resd <- as.data.frame(res)

In [19]:
write.table(x = resd, file = "Hepatocytes_out/Reports_otsu/QTL_overlaps/heps_atac_qtl_grn_phypter_res.csv")

In [15]:
resd[resd$ATAC.Hepatocytes_pval < 0.05,]

,ATAC.Hepatocytes_pval,ATAC.Hepatocytes_padj
,<dbl>,<dbl>
ATF3_direct_+/+,0.0276820431,0.15079429
CHD1_direct_+/+,0.0110304886,0.10392785
CHD2_direct_+/+,0.0012499475,0.03696273
CREB1_direct_+/+,0.0140116759,0.10976763
CREB5_direct_+/+,0.0397917276,0.17906277
CUX1_direct_+/+,0.0486752171,0.19049933
ELF1_direct_+/+,0.0128354267,0.10976763
EP300_direct_+/+,0.0014291584,0.03697947
FOXN3_direct_+/+,0.0114227092,0.10392785


In [16]:
resd[resd$ATAC.Hepatocytes_padj < 0.05,]

,ATAC.Hepatocytes_pval,ATAC.Hepatocytes_padj
,<dbl>,<dbl>
CHD2_direct_+/+,0.0012499475,0.03696273
EP300_direct_+/+,0.0014291584,0.03697947
FOXP1_direct_+/+,0.0003749048,0.02566944
GATAD2A_direct_+/+,0.0001483634,0.02566944
NFIA_direct_+/+,0.0016902908,0.03887669
NFIC_direct_+/+,0.0002577934,0.02566944
NR1H4_direct_+/+,0.0006200348,0.02566944
THRB_direct_+/+,0.0011675077,0.03696273
PPARD_direct_+/-,0.0005320302,0.02566944


In [49]:
summary(resd$ATAC.Hepatocytes_pval)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.9893  1.0000  1.0000  0.9997  1.0000  1.0000 

In [56]:
nrow(td[td$Region %in% regions,])

[1] 845

In [59]:
tid <- td[td$Region %in% regions,]
head(tid)

,Region,Gene,importance_R2G,rho_R2G,importance_x_rho,importance_x_abs_rho,TF,is_extended,eRegulon_name,Gene_signature_name,Region_signature_name,importance_TF2G,regulation,rho_TF2G,triplet_rank
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<dbl>,<int>
39,chr15:67943986-67944286,PIAS1,0.05924504,0.19669994,0.011653496,0.011653496,AR,False,AR_direct_+/+,AR_direct_+/+_(47g),AR_direct_+/+_(54r),0.8875806,1,0.28513007,10523
66,chr8:2152862-2153162,MYOM2,0.06180351,0.08220857,0.005080779,0.005080779,ATF3,False,ATF3_direct_+/+,ATF3_direct_+/+_(171g),ATF3_direct_+/+_(523r),1.2752246,1,0.08073879,19501
95,chr17:17435393-17435693,RASD1,0.09661725,0.25224359,0.024371083,0.024371083,ATF3,False,ATF3_direct_+/+,ATF3_direct_+/+_(171g),ATF3_direct_+/+_(523r),3.2653989,1,0.21004651,774
107,chr10:14532834-14533134,FAM107B,0.10820653,0.18000706,0.019477940,0.019477940,ATF3,False,ATF3_direct_+/+,ATF3_direct_+/+_(171g),ATF3_direct_+/+_(523r),1.0245517,1,0.21118817,12717
109,chr21:38797532-38797832,ETS2,0.04851221,0.22926921,0.011122357,0.011122357,ATF3,False,ATF3_direct_+/+,ATF3_direct_+/+_(171g),ATF3_direct_+/+_(523r),1.7102515,1,0.21760267,8803
116,chr17:79888947-79889247,CBX4,0.03952797,0.09414891,0.003721516,0.003721516,ATF3,False,ATF3_direct_+/+,ATF3_direct_+/+_(171g),ATF3_direct_+/+_(523r),2.8500979,1,0.10739487,8940


In [60]:
write.csv2(tid, file = "Hepatocytes_out/Reports_otsu/QTL_overlaps/Heps_atac_qtl_grn_intersect.csv")

In [61]:
### Now try coloc regions
t <- read.table("/tscc/lustre/ddn/scratch/welison/Liver_Share/result.depot/241213_WE_QTL_Feature_Beds/Hepatocyte_ATAC_coloc_features.bed", header = FALSE)
t <- na.omit(t)

In [62]:
regions <- paste0(t$V1, ":", t$V2, "-", t$V3)
head(regions)
length(regions)

[1] "chr1:16179136-16179436"  "chr1:16189049-16189288" 
[3] "chr1:16189980-16190280"  "chr11:61820741-61821041"
[5] "chr11:94132939-94133200" "chr12:52884847-52885147"

[1] 38

In [63]:
length(intersect(regions, td_sub$Region))

[1] 8

In [64]:
nrow(td[td$Region %in% regions,])

[1] 19

In [65]:
tid <- td[td$Region %in% regions,]
head(tid)

,Region,Gene,importance_R2G,rho_R2G,importance_x_rho,importance_x_abs_rho,TF,is_extended,eRegulon_name,Gene_signature_name,Region_signature_name,importance_TF2G,regulation,rho_TF2G,triplet_rank
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<dbl>,<int>
1240,chr11:61820741-61821041,DAGLA,0.01481841,0.12971243,0.0019221323,0.0019221323,BACH1,False,BACH1_direct_+/+,BACH1_direct_+/+_(593g),BACH1_direct_+/+_(1490r),0.6194808,1,0.12624605,30152
9538,chr2:232670328-232670628,EFHD1,0.01941836,0.12610035,0.0024486614,0.0024486614,FOXO1,False,FOXO1_direct_+/+,FOXO1_direct_+/+_(348g),FOXO1_direct_+/+_(585r),0.4338463,1,0.10074909,24378
10680,chr2:232668019-232668319,GIGYF2,0.01040544,0.06289372,0.0006544368,0.0006544368,FOXP2,False,FOXP2_direct_+/+,FOXP2_direct_+/+_(268g),FOXP2_direct_+/+_(288r),0.8867192,1,0.35766297,32912
10918,chr11:61820741-61821041,MYRF,0.02981133,0.06394151,0.0019061817,0.0019061817,GATAD2A,False,GATAD2A_direct_+/+,GATAD2A_direct_+/+_(134g),GATAD2A_direct_+/+_(143r),0.8726885,1,0.09441218,29565
10924,chr11:61820741-61821041,FADS3,0.02348805,0.08139535,0.0019118176,0.0019118176,GATAD2A,False,GATAD2A_direct_+/+,GATAD2A_direct_+/+_(134g),GATAD2A_direct_+/+_(143r),0.9311476,1,0.09028606,30859
11202,chr2:232636795-232637095,EFHD1,0.15151965,0.18963907,0.0287340444,0.0287340444,HNF4A,False,HNF4A_direct_+/+,HNF4A_direct_+/+_(217g),HNF4A_direct_+/+_(347r),1.2384544,1,0.12240683,596


In [66]:
write.csv2(tid, file = "Hepatocytes_out/Reports_otsu/QTL_overlaps/hep_coloc_atac_qtl_grn_intersect.csv")